# Model Selection
## CW1 - Regression Challenge

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import cross_val_score, KFold
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
import xgboost as xgb
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

np.random.seed(123)

trn = pd.read_csv('../data/raw/CW1_train.csv')
tst = pd.read_csv('../data/raw/CW1_test.csv')

## 1. Preprocessing

In [ ]:
def preprocess(df):
    df = df.copy()
    
    for col in ['x', 'y', 'z']:
        df[col] = df[col].replace(0, np.nan)
        df[col] = df[col].fillna(df[col].median())
    
    cut_map = {'Fair': 0, 'Good': 1, 'Very Good': 2, 'Premium': 3, 'Ideal': 4}
    color_map = {'J': 0, 'I': 1, 'H': 2, 'G': 3, 'F': 4, 'E': 5, 'D': 6}
    clarity_map = {'I1': 0, 'SI2': 1, 'SI1': 2, 'VS2': 3, 'VS1': 4, 'VVS2': 5, 'VVS1': 6, 'IF': 7}
    
    df['cut_ord'] = df['cut'].map(cut_map)
    df['color_ord'] = df['color'].map(color_map)
    df['clarity_ord'] = df['clarity'].map(clarity_map)
    
    df['depth_sq'] = df['depth'] ** 2
    df['depth_cb'] = df['depth'] ** 3
    df['depth_x_b3'] = df['depth'] * df['b3']
    df['depth_x_b1'] = df['depth'] * df['b1']
    df['depth_x_a1'] = df['depth'] * df['a1']
    df['depth_x_a4'] = df['depth'] * df['a4']
    df['depth_x_table'] = df['depth'] * df['table']
    df['b3_x_b1'] = df['b3'] * df['b1']
    df['b3_x_a1'] = df['b3'] * df['a1']
    df['a1_x_a4'] = df['a1'] * df['a4']
    df['b1_x_a1'] = df['b1'] * df['a1']
    df['b3_sq'] = df['b3'] ** 2
    df['b1_sq'] = df['b1'] ** 2
    df['a1_sq'] = df['a1'] ** 2
    df['a4_sq'] = df['a4'] ** 2
    df['table_sq'] = df['table'] ** 2
    df['volume'] = df['x'] * df['y'] * df['z']
    df['log_carat'] = np.log1p(df['carat'])
    df['log_price'] = np.log1p(df['price'])
    df['xy_ratio'] = df['x'] / df['y'].replace(0, np.nan).fillna(df['y'].median())
    
    df = df.drop(columns=['cut', 'color', 'clarity'])
    return df

trn_proc = preprocess(trn)
tst_proc = preprocess(tst)

X_trn = trn_proc.drop(columns=['outcome'])
y_trn = trn_proc['outcome']
X_tst = tst_proc

common_cols = X_trn.columns.intersection(X_tst.columns)
X_trn = X_trn[common_cols]
X_tst = X_tst[common_cols]

print(f"Features: {X_trn.shape[1]}")

## 2. Baseline Model Comparison

Comparing 8 regression models using 5-fold cross-validation with R² scoring.

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=123)

models = {
    'Linear Regression': LinearRegression(),
    'Ridge': Ridge(alpha=1.0),
    'Lasso': Lasso(alpha=0.1),
    'ElasticNet': ElasticNet(alpha=0.1, l1_ratio=0.5),
    'Random Forest': RandomForestRegressor(n_estimators=200, max_depth=10, random_state=123, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=200, max_depth=5, learning_rate=0.1, random_state=123),
    'XGBoost': xgb.XGBRegressor(n_estimators=200, max_depth=5, learning_rate=0.1, random_state=123, n_jobs=-1),
    'LightGBM': lgb.LGBMRegressor(n_estimators=200, max_depth=5, learning_rate=0.1, random_state=123, n_jobs=-1, verbose=-1),
}

results = {}
print(f"{'Model':25s}  {'Mean R²':>10s}  {'Std':>8s}")
print("-" * 48)
for name, model in models.items():
    scores = cross_val_score(model, X_trn, y_trn, cv=kf, scoring='r2', n_jobs=-1)
    results[name] = scores
    print(f"{name:25s}  {scores.mean():10.4f}  {scores.std():8.4f}")

## 3. Model Comparison Visualisation

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

names = list(results.keys())
means = [results[n].mean() for n in names]
stds = [results[n].std() for n in names]

colors = ['#2ecc71' if m == max(means) else '#3498db' for m in means]
bars = ax.barh(names, means, xerr=stds, color=colors, edgecolor='black', alpha=0.8, capsize=5)
ax.set_xlabel('R² Score (5-Fold CV)')
ax.set_title('Model Comparison')
ax.axvline(x=0, color='grey', linestyle='--', alpha=0.5)

for i, (m, s) in enumerate(zip(means, stds)):
    ax.text(m + s + 0.005, i, f'{m:.4f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('../reports/figures/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Results Summary

Tree-based models (Random Forest, Gradient Boosting, XGBoost, LightGBM) significantly outperform linear models. This makes sense given that `depth` has a strong non-linear relationship with outcome.

Linear models achieve R² ≈ 0.23–0.28, while tree-based models reach R² ≈ 0.45–0.46. We will proceed with **XGBoost** and **LightGBM** for hyperparameter tuning in Phase 3.